# Module 4: Evaluation

In [2]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py

--2026-07-07 08:56:33--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py
Loaded CA certificate '/etc/ssl/certs/ca-certificates.crt'
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3073 (3.0K) [text/plain]
Saving to: ‘evaluation_utils.py.1’

evaluation_utils.py 100%[===================>]   3.00K  --.-KB/s    in 0s      

2026-07-07 08:56:34 (92.8 MB/s) - ‘evaluation_utils.py.1’ saved [3073/3073]



In [3]:
from ingest import load_faq_data

In [4]:
documents = load_faq_data()

In [5]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

As the LLM course is newer we will use the FAQs from just this course as there are far fewer FAQs.

In [6]:
from pydantic import BaseModel

In [7]:
class Questions(BaseModel):
    questions: list[str]

In [8]:
data_generation_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

### OpenAI Config

In [9]:
import os
from dotenv import dotenv_values, load_dotenv

secrets_dir = os.path.expanduser("~/Documents/.secrets/llm-zoomcamp/")

config = {
    **dotenv_values(secrets_dir + "/.env.openai"),
}

api_key = config.get("OPENAI_API_KEY")

In [10]:
from openai import OpenAI
openai_client = OpenAI(api_key=api_key)

## Basics of generating questions from a single document

In [11]:
import json

In [12]:
doc = documents_llm[0]

In [13]:
user_prompt = json.dumps(doc)

In [14]:
messages = [
    {"role": "developer", "content": data_generation_instructions},
    {"role": "user", "content": user_prompt}
]

We need to use the .parse instead of .create from the open ai responses api

In [15]:
response = openai_client.responses.parse(
        model='gpt-5.4-mini',
        input=messages,
        text_format=Questions
    )

In [16]:
result = response.output_parsed
print(result)

questions=['I just found this course — is it still possible to join now?', 'Can I enroll late and still take the course?', 'If I start the course after it already began, will I still be able to participate?', 'Is joining the course after it starts allowed, or am I too late?', 'I missed the beginning of the course; can I still sign up and get a certificate?']


## Generate for a single document using utils

In [17]:
from evaluation_utils import llm_structured, calc_price

In [18]:
result, usage = llm_structured(
    openai_client,
    data_generation_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course—am I still able to join now?', 'Can I join late, or is it too late to start the course?', 'If I start the course now, can I still get a certificate?', 'What do I need to do to be eligible for the certificate if I join now?', 'Is there still time to submit the project and receive a certificate?']


In [19]:
calc_price(usage)

{'input_cost': 0.00015525, 'output_cost': 0.0004005, 'total_cost': 0.00055575}

In [20]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course—am I still able to join now?',
  'document': '74eb249bbf'},
 {'question': 'Can I join late, or is it too late to start the course?',
  'document': '74eb249bbf'},
 {'question': 'If I start the course now, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for the certificate if I join now?',
  'document': '74eb249bbf'},
 {'question': 'Is there still time to submit the project and receive a certificate?',
  'document': '74eb249bbf'}]

## Generating questions for all documents

In [25]:
import pandas as pd
from evaluation_utils import llm_structured_retry
from tqdm.auto import tqdm

In [22]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [24]:
def generate_ground_truth(doc: dict[str]):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_generation_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [26]:
generate_ground_truth(doc)

([{'question': 'Can I still join the course if I just found it now, or is it too late?',
   'document': '74eb249bbf'},
  {'question': 'Is it okay to start the course late and still take part?',
   'document': '74eb249bbf'},
  {'question': 'If I join the course after it has started, can I still get a certificate?',
   'document': '74eb249bbf'},
  {'question': 'What do I need to do to be eligible for the certificate if I’m joining late?',
   'document': '74eb249bbf'},
  {'question': 'Does joining the course now still count, and is there any deadline for the project?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=100, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=307))

In [27]:
ground_truth = []
usages = []

for doc in tqdm(documents_llm[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [28]:
ground_truth

[{'question': 'I just found this course — can I still join now, or is it too late?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, will I still be able to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to qualify for the certificate if I started after the course began?',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to enroll now, and how does that affect project submission deadlines?',
  'document': '74eb249bbf'},
 {'question': 'Can I still participate in the course even though I discovered it after it started?',
  'document': '74eb249bbf'},
 {'question': 'I signed up for the LLM Zoomcamp, but I never got any confirmation email — does that mean my registration didn’t go through?',
  'document': '977bf7786c'},
 {'question': 'Do I actually need an acceptance or confirmation email before I can start the LLM Zoomcamp homework?',
  'document': '977bf7786c'},
 {'question': 'If I didn’t register for the course yet, can I still beg

## Parallel processing

With our small sample it took 9seconds to generate all of the questions. If can do this in parallel, this will be more scalable.

In [29]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [30]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents_llm, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]

In [32]:
results[0]

([{'question': "Can I still join the course if I'm late to it?",
   'document': '74eb249bbf'},
  {'question': 'If I join the course now, can I still get a certificate?',
   'document': '74eb249bbf'},
  {'question': 'Do I need to finish the project before submissions close to get the certificate?',
   'document': '74eb249bbf'},
  {'question': 'Is it too late to start this course, or can I still enroll?',
   'document': '74eb249bbf'},
  {'question': 'What’s the deadline for project submission if I want a certificate?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=85, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=292))

In [31]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

515

In [33]:
ground_truth[3]

{'question': 'Is it too late to start this course, or can I still enroll?',
 'document': '74eb249bbf'}

In [34]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.07839524999999997

### Save ground truths to csv

In [35]:
df_ground_truth = pd.DataFrame(ground_truth)

In [36]:
df_ground_truth.head()

,question,document
0,Can I still join the course if I'm late to it?,74eb249bbf
1,"If I join the course now, can I still get a ce...",74eb249bbf
2,Do I need to finish the project before submiss...,74eb249bbf
3,"Is it too late to start this course, or can I ...",74eb249bbf
4,What’s the deadline for project submission if ...,74eb249bbf


In [37]:
df_ground_truth.to_csv("./data/ground_truth.csv", index=False)